# Image captioning — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def conv2d(x,k,pad=0,stride=1,dilation=1):
    x=np.asarray(x,float); k=np.asarray(k,float)
    if pad: x=np.pad(x,pad)
    kh,kw=k.shape; dh=(kh-1)*dilation+1; dw=(kw-1)*dilation+1; out=[]
    for i in range(0,x.shape[0]-dh+1,stride):
        row=[]
        for j in range(0,x.shape[1]-dw+1,stride): row.append(float(np.sum(x[i:i+dh:dilation,j:j+dw:dilation]*k)))
        out.append(row)
    return np.array(out)
def iou(a,b):
    ax1,ay1,ax2,ay2=a; bx1,by1,bx2,by2=b
    ix1,iy1=max(ax1,bx1),max(ay1,by1); ix2,iy2=min(ax2,bx2),min(ay2,by2)
    inter=max(0,ix2-ix1)*max(0,iy2-iy1); union=(ax2-ax1)*(ay2-ay1)+(bx2-bx1)*(by2-by1)-inter
    return inter/union if union else 0
def softmax(z):
    z=np.asarray(z,float); e=np.exp(z-z.max()); return e/e.sum()
def show(a,title,cmap='viridis'):
    plt.figure(figsize=(4,3)); plt.imshow(a,cmap=cmap); plt.colorbar(); plt.title(title)
print('setup ok')

## Image captioning
Tiny CPU-only synthetic arrays for Image captioning: no downloads, no pretrained models, and every cell ends with an assert.

In [ ]:
# 1) image patches become tokens
img=np.arange(16).reshape(4,4); patches=np.array([img[i:i+2,j:j+2].ravel() for i in range(0,4,2) for j in range(0,4,2)]); plt.figure(figsize=(4,3)); plt.imshow(patches); plt.title('patch tokens')
assert patches.shape==(4,4)

In [ ]:
# 2) positional information distinguishes equal patches
tokens=np.ones((4,3)); pos=np.arange(12).reshape(4,3)/100; x=tokens+pos; plt.figure(figsize=(4,3)); plt.imshow(x); plt.title('positions')
assert not np.allclose(x[0],x[1])

In [ ]:
# 3) attention weights are normalized similarities
Q=np.array([[1.,0.],[0.,1.]]); K=Q.copy(); V=np.array([[2.,0.],[0.,3.]]); A=np.exp(Q@K.T); A=A/A.sum(1,keepdims=True); out=A@V; plt.figure(figsize=(3,3)); plt.imshow(A); plt.colorbar(); plt.title('attention')
assert out[0,0]>out[0,1]

In [ ]:
# 4) contrastive learning aligns positive views
v1=np.array([1.1,2.,2.9]); v2=np.array([.9,2.05,3.1]); cos=v1@v2/(np.linalg.norm(v1)*np.linalg.norm(v2)); plt.figure(figsize=(4,3)); plt.bar(['cosine'],[cos]); plt.ylim(0,1); plt.title('positive pair')
assert cos>.99

In [ ]:
# 5) caption decoder turns logits into next-word probabilities
logits=np.array([.1,2.0,.5]); p=softmax(logits); plt.figure(figsize=(4,3)); plt.bar(['a','dog','runs'],p); plt.title('next token')
assert p.argmax()==1 and abs(p.sum()-1)<1e-12